# Notebook 03: SetFit
Runs setfit few-shot multilabel experiments with resumable result persistence.


In [ ]:
# install setfit stack and import core libraries
!pip install -q -U "setfit>=1.1.1" "transformers>=4.46,<5" "huggingface_hub>=0.26.2" "accelerate>=0.30" datasets

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.metrics import f1_score, classification_report
import torch
import setfit
import transformers
import huggingface_hub
from setfit import SetFitModel, Trainer, TrainingArguments

try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = "/content/drive/MyDrive/cs4120_project/"
except Exception:
    SAVE_DIR = None

print(f"PyTorch: {torch.__version__}")
print(f"SetFit: {setfit.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"HuggingFace Hub: {huggingface_hub.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 124.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 61.9 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Mounted at /content/drive
PyTorch: 2.10.0+cu128
SetFit: 1.1.3
Transformers: 4.57.6
HuggingFace Hub: 0.36.2
GPU available: True
GPU: NVIDIA H100 80GB HBM3


In [ ]:
# clone repo and enter notebooks directory
!git clone https://github.com/phisomni-edu/cs4120-project /content/cs4120-project
%cd /content/cs4120-project/notebooks

Cloning into '/content/cs4120-project'...
remote: Enumerating objects: 236, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 236 (delta 20), reused 28 (delta 18), pack-reused 200 (from 1)
Receiving objects: 100% (236/236), 8.74 MiB | 13.95 MiB/s, done.
Resolving deltas: 100% (130/130), done.
/content/cs4120-project/notebooks


In [ ]:
# resolve paths, load splits, and parse labels
import ast
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.preprocessing import MultiLabelBinarizer

# Resolve repo root so src/ imports work in both local Jupyter and Colab
repo_candidates = [Path.cwd(), Path.cwd().parent]
repo_root = None
for candidate in repo_candidates:
    if (candidate / "src" / "evaluate.py").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not find repo root containing src/evaluate.py")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.evaluate import evaluate_run, save_evaluation_outputs

def _parse_labels_column(df, label_col="labels"):
    df = df.copy()

    def _parse_one(value):
        if isinstance(value, list):
            return [int(x) for x in value]
        if isinstance(value, np.ndarray):
            return [int(x) for x in value.tolist()]
        if isinstance(value, str):
            s = value.strip()
            if s.startswith('[') and s.endswith(']'):
                body = s[1:-1].strip()
                if body == "":
                    return []
                if ',' in body:
                    tokens = [t.strip() for t in body.split(',') if t.strip()]
                else:
                    tokens = [t for t in body.split() if t]
                return [int(t) for t in tokens]
            return [int(x) for x in ast.literal_eval(s)]
        return value

    df[label_col] = df[label_col].apply(_parse_one)
    return df

def _resolve_data_dir():
    # Local notebooks usually run from notebooks/, so data is often ../data
    for candidate in [repo_root / "data", Path("../data"), Path("data")]:
        if (candidate / "test.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find data directory containing test.csv")

def _get_label_names():
    dataset = load_dataset("google-research-datasets/go_emotions", "simplified")
    return dataset["train"].features["labels"].feature.names

def evaluate_setfit_predictions(
    y_pred_binary,
    *,
    data_fraction,
    seed,
    method="setfit",
    data_dir=None,
    output_dir=None,
):
    """
    Evaluate SetFit predictions on the shared test split and save standard outputs.

    y_pred_binary: binary matrix (n_samples, n_labels) aligned with test.csv rows.
    """
    data_dir = Path(data_dir) if data_dir else _resolve_data_dir()
    output_dir = Path(output_dir) if output_dir else (repo_root / "results")

    test_df = pd.read_csv(data_dir / "test.csv")
    test_df = _parse_labels_column(test_df, label_col="labels")

    label_names = _get_label_names()
    mlb = MultiLabelBinarizer(classes=list(range(len(label_names))))
    mlb.fit([list(range(len(label_names)))])
    y_true_binary = mlb.transform(test_df["labels"])

    evaluation = evaluate_run(
        method=method,
        data_fraction=float(data_fraction),
        seed=int(seed),
        label_names=label_names,
        y_true=y_true_binary,
        y_pred=y_pred_binary,
    )

    paths = save_evaluation_outputs(evaluation, method=method, output_dir=output_dir)
    return evaluation, paths

print("Shared evaluation helpers loaded.")
print("After generating SetFit predictions, call evaluate_setfit_predictions(y_pred_binary, data_fraction=..., seed=...)")


Shared evaluation helpers loaded.
After generating SetFit predictions, call evaluate_setfit_predictions(y_pred_binary, data_fraction=..., seed=...)


In [ ]:
# experiment grid + model config
METHOD_NAME = "setfit"
MODEL_NAME = "sentence-transformers/paraphrase-mpnet-base-v2"

DATA_FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
SEEDS = [42, 7, 21]

TRAIN_FILE_BY_FRACTION = {
    0.01: "train_1pct.csv",
    0.05: "train_5pct.csv",
    0.10: "train_10pct.csv",
    0.25: "train_25pct.csv",
    0.50: "train_50pct.csv",
    1.00: "train.csv",
}

SETFIT_CONFIG = {
    "num_iterations": 20,
    "num_epochs": 1,
    "batch_size": 16,
    "body_learning_rate": 2e-5,
    "head_learning_rate": 1e-2,
}

RESULTS_DIR = repo_root / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ENABLE_CHECKPOINT_SAVE = ("SAVE_DIR" in globals()) and bool(SAVE_DIR)
if ENABLE_CHECKPOINT_SAVE:
    CHECKPOINT_ROOT = Path(SAVE_DIR) / "checkpoints" / "setfit"
    CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

print("Config loaded.")
print("Results dir:", RESULTS_DIR)


Config loaded.
Results dir: /content/cs4120-project/results


In [ ]:
# define setfit training/evaluation helper functions
import gc
import os
import random
from datasets import Dataset

# Disable W&B prompts in notebook runs
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

def _set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        pass

def _choose_text_col(df):
    for col in ["text_clean_transformer", "text"]:
        if col in df.columns:
            return col
    raise ValueError("No text column found. Expected one of: text_clean_transformer, text")

# convert label-id lists into fixed-width multilabel vectors
def _to_binary_matrix(label_lists, n_labels):
    mlb = MultiLabelBinarizer(classes=list(range(n_labels)))
    mlb.fit([list(range(n_labels))])
    return mlb.transform(label_lists)

def _predictions_to_binary(preds, n_labels):
    # Case 1: already matrix-like
    try:
        arr = np.asarray(preds)
        if arr.ndim == 2 and arr.shape[1] == n_labels:
            return (arr > 0).astype(int)
    except Exception:
        pass

    # Case 2: sequence of predicted label-id collections
    out = np.zeros((len(preds), n_labels), dtype=int)
    for i, pred in enumerate(preds):
        if isinstance(pred, (list, tuple, set, np.ndarray)):
            for label_id in pred:
                try:
                    idx = int(label_id)
                    if 0 <= idx < n_labels:
                        out[i, idx] = 1
                except Exception:
                    continue
        else:
            try:
                idx = int(pred)
                if 0 <= idx < n_labels:
                    out[i, idx] = 1
            except Exception:
                continue
    return out

def _build_setfit_model(model_name):
    try:
        return SetFitModel.from_pretrained(model_name, multi_target_strategy="one-vs-rest")
    except TypeError:
        return SetFitModel.from_pretrained(model_name)

# handle trainer api differences across setfit versions
def _build_trainer(model, train_dataset, eval_dataset, cfg):
    last_exc = None

    try:
        args = TrainingArguments(
            batch_size=cfg["batch_size"],
            num_epochs=cfg["num_epochs"],
            num_iterations=cfg["num_iterations"],
            body_learning_rate=cfg["body_learning_rate"],
            head_learning_rate=cfg["head_learning_rate"],
            report_to="none",
        )
        return Trainer(
            model=model,
            args=args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            column_mapping={"text": "text", "label": "label"},
        )
    except Exception as exc:
        last_exc = exc

    try:
        return Trainer(
            model=model,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            batch_size=cfg["batch_size"],
            num_epochs=cfg["num_epochs"],
            num_iterations=cfg["num_iterations"],
            column_mapping={"text": "text", "label": "label"},
        )
    except Exception as exc:
        last_exc = exc

    raise RuntimeError(f"Could not construct SetFit Trainer with known APIs: {last_exc}")

# run one fraction/seed and return shared-eval outputs
def run_single_setfit_experiment(data_fraction, seed):
    data_dir = _resolve_data_dir()

    train_filename = TRAIN_FILE_BY_FRACTION[float(data_fraction)]
    train_path = data_dir / train_filename
    test_path = data_dir / "test.csv"

    if not train_path.exists():
        raise FileNotFoundError(f"Missing train file for fraction={data_fraction}: {train_path}")
    if not test_path.exists():
        raise FileNotFoundError(f"Missing test file: {test_path}")

    train_df = _parse_labels_column(pd.read_csv(train_path), label_col="labels")
    test_df = _parse_labels_column(pd.read_csv(test_path), label_col="labels")

    text_col = _choose_text_col(train_df)
    label_names = _get_label_names()
    n_labels = len(label_names)

    y_train = _to_binary_matrix(train_df["labels"], n_labels)
    y_test = _to_binary_matrix(test_df["labels"], n_labels)

    train_dataset = Dataset.from_dict({
        "text": train_df[text_col].fillna("").astype(str).tolist(),
        "label": y_train.tolist(),
    })
    eval_dataset = Dataset.from_dict({
        "text": test_df[text_col].fillna("").astype(str).tolist(),
        "label": y_test.tolist(),
    })

    _set_global_seed(int(seed))
    model = _build_setfit_model(MODEL_NAME)
    trainer = _build_trainer(model, train_dataset, eval_dataset, SETFIT_CONFIG)
    trainer.train()

    raw_preds = model.predict(test_df[text_col].fillna("").astype(str).tolist())
    y_pred = _predictions_to_binary(raw_preds, n_labels)

    evaluation = evaluate_run(
        method=METHOD_NAME,
        data_fraction=float(data_fraction),
        seed=int(seed),
        label_names=label_names,
        y_true=y_test,
        y_pred=y_pred,
    )

    if ENABLE_CHECKPOINT_SAVE and float(data_fraction) == 1.00 and SAVE_DIR:
        ckpt_dir = CHECKPOINT_ROOT / f"seed_{seed}"
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(str(ckpt_dir))

    try:
        del trainer
        del model
    except Exception:
        pass
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

    return evaluation


## 5. Resumable Runner (Crash-Safe)

Loads existing result files, tracks completed runs, and persists after every successful run.


In [ ]:
# resumable csv outputs
OVERALL_PATH = RESULTS_DIR / "setfit_overall.csv"
PER_CLASS_PATH = RESULTS_DIR / "setfit_per_class.csv"
README_PATH = RESULTS_DIR / "setfit_results.csv"
ERRORS_PATH = RESULTS_DIR / "setfit_errors.csv"

def _safe_read_csv(path):
    if not path.exists():
        return pd.DataFrame()

    try:
        if path.stat().st_size == 0:
            return pd.DataFrame()
        return pd.read_csv(path)
    except (pd.errors.EmptyDataError, FileNotFoundError):
        return pd.DataFrame()

overall_df = _safe_read_csv(OVERALL_PATH)
per_class_df = _safe_read_csv(PER_CLASS_PATH)
errors_df = _safe_read_csv(ERRORS_PATH)

def _dedup_overall(df):
    if df.empty:
        return df
    return (
        df.sort_values(["method", "seed", "data_fraction"])
          .drop_duplicates(subset=["method", "seed", "data_fraction"], keep="last")
          .reset_index(drop=True)
    )

def _dedup_per_class(df):
    if df.empty:
        return df
    return (
        df.sort_values(["method", "seed", "data_fraction", "emotion"])
          .drop_duplicates(subset=["method", "seed", "data_fraction", "emotion"], keep="last")
          .reset_index(drop=True)
    )

def _persist_results():
    global overall_df, per_class_df, errors_df

    overall_df = _dedup_overall(overall_df)
    per_class_df = _dedup_per_class(per_class_df)

    overall_df.to_csv(OVERALL_PATH, index=False)
    per_class_df.to_csv(PER_CLASS_PATH, index=False)

    if not per_class_df.empty:
        readme_df = per_class_df[["method", "data_fraction", "seed", "emotion", "f1", "precision", "recall"]].copy()
        readme_df.to_csv(README_PATH, index=False)

    if not errors_df.empty:
        errors_df.to_csv(ERRORS_PATH, index=False)

def _completed_pairs():
    if overall_df.empty:
        return set()
    return set((float(r.data_fraction), int(r.seed)) for r in overall_df[["data_fraction", "seed"]].itertuples(index=False))

def show_setfit_progress():
    done = _completed_pairs()
    expected = [(float(f), int(s)) for s in SEEDS for f in DATA_FRACTIONS]
    pending = [p for p in expected if p not in done]

    print(f"Completed runs: {len(done)} / {len(expected)}")
    if pending:
        print("Pending:", pending)
    else:
        print("No pending runs.")

    if not overall_df.empty:
        display(overall_df.sort_values(["seed", "data_fraction"]).tail(10))

show_setfit_progress()


Completed runs: 11 / 18
Pending: [(1.0, 7), (0.01, 21), (0.05, 21), (0.1, 21), (0.25, 21), (0.5, 21), (1.0, 21)]


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,method,data_fraction,seed,accuracy,macro_f1,micro_f1,hamming_loss
1,setfit,0.05,7,0.390271,0.374964,0.500129,0.038360
2,setfit,0.10,7,0.407039,0.405670,0.517747,0.038090
3,setfit,0.25,7,0.425650,0.438534,0.536629,0.037379
4,setfit,0.50,7,0.432467,0.448320,0.540179,0.037129
5,setfit,0.01,42,0.336650,0.264521,0.461088,0.037550
6,setfit,0.05,42,0.390271,0.374964,0.500129,0.038360
7,setfit,0.10,42,0.407039,0.405670,0.517747,0.038090
8,setfit,0.25,42,0.425650,0.438534,0.536629,0.037379
9,setfit,0.50,42,0.432467,0.448320,0.540179,0.037129
10,setfit,1.00,42,0.444260,0.440057,0.551101,0.035958


## 6. Run One Job or Small Batch

Use these helpers to run one `(fraction, seed)` at a time or a bounded number of pending runs.


In [ ]:
# run pending jobs with crash-safe persistence
def run_and_persist_one(data_fraction, seed, *, force=False):
    global overall_df, per_class_df, errors_df

    key = (float(data_fraction), int(seed))
    if (not force) and key in _completed_pairs():
        print(f"Skipping completed run: {key}")
        return

    print(f"Running SetFit: fraction={data_fraction}, seed={seed}")
    try:
        eval_out = run_single_setfit_experiment(data_fraction, seed)

        if not overall_df.empty:
            overall_df = overall_df[~((overall_df["data_fraction"].astype(float) == float(data_fraction)) & (overall_df["seed"].astype(int) == int(seed)))]
        if not per_class_df.empty:
            per_class_df = per_class_df[~((per_class_df["data_fraction"].astype(float) == float(data_fraction)) & (per_class_df["seed"].astype(int) == int(seed)))]

        overall_df = pd.concat([overall_df, eval_out["overall"]], ignore_index=True) if not overall_df.empty else eval_out["overall"].copy()
        per_class_df = pd.concat([per_class_df, eval_out["per_class"]], ignore_index=True) if not per_class_df.empty else eval_out["per_class"].copy()

        _persist_results()

        macro_f1 = float(eval_out["overall"].iloc[0]["macro_f1"])
        print(f"Completed and saved: macro_f1={macro_f1:.4f}")

    except Exception as exc:
        err_row = pd.DataFrame([{"seed": int(seed), "data_fraction": float(data_fraction), "error": str(exc)}])
        errors_df = pd.concat([errors_df, err_row], ignore_index=True) if not errors_df.empty else err_row
        _persist_results()
        print(f"FAILED: {exc}")

# run only incomplete jobs for crash-safe continuation
def run_pending(*, fractions=None, seeds=None, max_runs=None):
    fracs = [float(f) for f in (fractions if fractions is not None else DATA_FRACTIONS)]
    seed_list = [int(s) for s in (seeds if seeds is not None else SEEDS)]

    planned = [(f, s) for s in seed_list for f in fracs]
    pending = [p for p in planned if p not in _completed_pairs()]

    if max_runs is not None:
        pending = pending[: int(max_runs)]

    print(f"Pending selected runs: {len(pending)}")
    for f, s in pending:
        run_and_persist_one(f, s)

    show_setfit_progress()


In [ ]:
# run_and_persist_one(0.50, 42)

# run_pending(fractions=[0.25])

# run_pending(max_runs=2)

run_pending()

show_setfit_progress()


Pending selected runs: 7
Running SetFit: fraction=1.0, seed=7


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reus

README.md: 0.00B [00:00, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,method,data_fraction,seed,accuracy,macro_f1,micro_f1,hamming_loss
8,setfit,0.10,21,0.407039,0.405670,0.517747,0.038090
9,setfit,0.25,21,0.425650,0.438534,0.536629,0.037379
10,setfit,0.50,21,0.432467,0.448320,0.540179,0.037129
11,setfit,1.00,21,0.444260,0.440057,0.551101,0.035958
12,setfit,0.01,42,0.336650,0.264521,0.461088,0.037550
13,setfit,0.05,42,0.390271,0.374964,0.500129,0.038360
14,setfit,0.10,42,0.407039,0.405670,0.517747,0.038090
15,setfit,0.25,42,0.425650,0.438534,0.536629,0.037379
16,setfit,0.50,42,0.432467,0.448320,0.540179,0.037129
17,setfit,1.00,42,0.444260,0.440057,0.551101,0.035958


Completed runs: 18 / 18
No pending runs.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,method,data_fraction,seed,accuracy,macro_f1,micro_f1,hamming_loss
8,setfit,0.10,21,0.407039,0.405670,0.517747,0.038090
9,setfit,0.25,21,0.425650,0.438534,0.536629,0.037379
10,setfit,0.50,21,0.432467,0.448320,0.540179,0.037129
11,setfit,1.00,21,0.444260,0.440057,0.551101,0.035958
12,setfit,0.01,42,0.336650,0.264521,0.461088,0.037550
13,setfit,0.05,42,0.390271,0.374964,0.500129,0.038360
14,setfit,0.10,42,0.407039,0.405670,0.517747,0.038090
15,setfit,0.25,42,0.425650,0.438534,0.536629,0.037379
16,setfit,0.50,42,0.432467,0.448320,0.540179,0.037129
17,setfit,1.00,42,0.444260,0.440057,0.551101,0.035958
